# Profile Chatbot

OpenAI chat + tool calling + Gradio UI, based on [4_lab4.ipynb](https://github.com/ed-donner/agents/blob/main/1_foundations/4_lab4.ipynb).

**Before running:**
1. Put `OPENAI_API_KEY` in the parent `.env`
2. Edit `summary.txt` about yourself
3. Optionally add `linkedin.pdf` next to this notebook (LinkedIn → Resources → Save to PDF)

**Modular scripts** (same logic): `agent/context.py`, `agent/tools.py`, `profile_chatbot.py` — run with `python profile_chatbot.py`.

Policy harness / live gate live in `../2_harness/`.

In [ ]:
# imports

from pathlib import Path
import json
import os

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader

In [ ]:
# load env from parent project folder, then init OpenAI

HERE = Path.cwd()
load_dotenv(HERE.parent / ".env", override=True)
load_dotenv(HERE / ".env", override=True)  # local override if present

openai = OpenAI()
MODEL_NAME = "gpt-5.4-mini"  # same as upstream lab; change if your account needs another model

print("OPENAI_API_KEY:", "ok" if os.getenv("OPENAI_API_KEY") else "missing")

## Tools

The model can call these during a turn. For this practice they just print to the notebook output so you can see the agent loop.

In [ ]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    print(f"[tool] record_user_details: {name=} {email=} {notes=}")
    return "OK"


def record_recommended_company(company_name):
    print(f"[tool] record_recommended_company: {company_name=}")
    return "OK"

In [ ]:
record_user_details_json = {
    "name": "record_user_details",
    "description": (
        "Use this tool to record that a user is interested in being in touch "
        "and provided an email address"
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"},
            "name": {"type": "string", "description": "The user's name, if they provided it"},
            "notes": {
                "type": "string",
                "description": "Any additional info about the conversation worth recording",
            },
        },
        "required": ["email"],
        "additionalProperties": False,
    },
}

record_recommended_company_json = {
    "name": "record_recommended_company",
    "description": "Use this tool to record company name that user recommend",
    "parameters": {
        "type": "object",
        "properties": {
            "company_name": {
                "type": "string",
                "description": "The company name that user recommend",
            },
        },
        "required": ["company_name"],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_recommended_company_json},
]

tool_map = {
    "record_user_details": record_user_details,
    "record_recommended_company": record_recommended_company,
}

In [ ]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = tool_map.get(tool_name)
        result = tool(**arguments) if tool else f"Unknown tool: {tool_name}"
        results.append(
            {
                "role": "tool",
                "content": json.dumps(result),
                "tool_call_id": tool_call.id,
            }
        )
    return results

## Context (summary + LinkedIn PDF)

In [ ]:
summary_path = HERE / "summary.txt"
linkedin_path = HERE / "linkedin.pdf"

summary = summary_path.read_text(encoding="utf-8") if summary_path.exists() else ""

linkedin = ""
if linkedin_path.exists():
    reader = PdfReader(str(linkedin_path))
    for page in reader.pages:
        text = page.extract_text()
        if text:
            linkedin += text
else:
    print("linkedin.pdf not found — chatbot will rely on summary.txt only")

print(f"summary chars: {len(summary)}")
print(f"linkedin chars: {len(linkedin)}")

In [ ]:
system_prompt = f"""
# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.

If the user recommends a company, use your tool to record the company name.

IMPORTANT:
If you don't know the answer, tell the user that you don't know. Never make up an answer.
""".strip()

## Chat loop + Gradio

Agent flow: user message → OpenAI (with tools) → if `tool_calls`, run tools and call again → return final reply.

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [
        {"role": "user", "content": message}
    ]
    response = openai.chat.completions.create(
        model=MODEL_NAME, messages=messages, tools=tools
    )
    while response.choices[0].finish_reason == "tool_calls":
        msg = response.choices[0].message
        results = handle_tool_calls(msg.tool_calls)
        messages.append(msg)
        messages.extend(results)
        response = openai.chat.completions.create(
            model=MODEL_NAME, messages=messages, tools=tools
        )
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)